In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from enum import IntEnum
from functools import lru_cache
from pathlib import Path

### Загрузка данных

In [ ]:
parquet_path = Path('../results_new.parquet')
df = pd.read_parquet(parquet_path) if parquet_path.exists() else pd.read_csv('../results_new.csv', comment='#')
print(df.shape)
df.head(3)

### Округление точности

In [ ]:
df['delta1'] = df['delta1'].round(6)
df['delta2'] = df['delta2'].round(6)
df['eps'] = df['eps'].round(6)

In [ ]:
COUPLING_NAMES = ['Инерционная', 'Инерционная (норм.)', 'Диссипативная', 'Диссипативная (норм.)']
# Подписи на графиках русские, а имена файлов — латиницей: кириллица в путях ломает
# \includegraphics в Overleaf.
COUPLING_SLUGS = ['inertial', 'inertialnorm', 'dissipative', 'dissipativenorm']
epsilons = sorted(df['eps'].unique())

class CouplingType(IntEnum):
    Inertial = 0
    InertialNorm = 1
    Dissipative = 2
    DissipativeNorm = 3

# Порог захвата. Сверху ограничен разрешением конечного окна: при |s| >= 2π/T_obs
# разность фаз за время наблюдения совершает хотя бы один полный проскок 2π —
# это несовместимо с определением захвата (ограниченность разности фаз).
# Снизу — остаточным наклоном в ядре языка (численный ноль, ≤ 2.5e-3).
# Обе границы задаются только длиной окна и не зависят ни от ε, ни от типа связи,
# поэтому θ здесь одна на всё (прежняя таблица 3×4 подобранных значений убрана).
T_OBS = 1600 - 240            # T - t_trans, см. simulation/experiments/multistability/config.yaml
THETA = 2 * np.pi / T_OBS     # ≈ 0.0046

# Максимум доли н.у. вне доминирующего режима. Мода — самое частое значение, а не
# большинство: при 30 н.у. и трёх равных кодах (10/10/10) мода = 1/3, вне моды = 2/3.
# Наблюдаемый максимум по данным — 0.667, поэтому шкала до 0.7, а не до 1.
DIS_MAX = 0.7

# Код строится по всем трём парам, но s12 НЕ является независимым измерением:
# из тождества (φ0-φ1) = (φ0-φ2) - (φ1-φ2) и линейности МНК-наклона следует
# s01 = s02 - s12 ТОЧНО (проверено: выполняется с машинной точностью на всех строках).
# Независимых величин две, а не три.
#
# Отсюда два следствия.
# 1) Комбинация «ровно две пары из трёх заперты» динамически невозможна: если Ω01=0
#    и Ω02=0, то Ω12=0 автоматически. Но порог заменяет «=0» на «<θ», а это свойство
#    при вычитании не сохраняется: |s12| <= |s01| + |s02| < 2θ. Третий наклон может
#    попасть в [θ, 2θ) и не пройти порог. Такие точки (>=2 бита из 3) сводим в один
#    код «полная».
# 2) Захват пары 1↔2 при незапертом хабе — это НЕ remote synchronization. Он держится
#    только на линии δ1=δ2, где листья идентичны и работает перестановочная симметрия
#    сети; ширина по расстройке там не превышает шага сетки. Вблизи центра полоса шире,
#    но лишь потому, что оба листа близки к захвату с хабом и обе частоты малы.
REGIME_LABELS = ['нет синхр.', '0↔1', '0↔2', '1↔2', 'полная (≥2 из 3 пар)']
REGIME_COLORS = ['#eeeeee', '#4477aa', '#66ccee', '#ccbb44', '#8b1a89']

print('epsilons:', epsilons)
print('coupling types:', sorted(df['coupling_type'].unique()))
print('runs per point:', df.groupby(['delta1', 'delta2', 'eps', 'coupling_type']).size().unique())
print(f'T_obs = {T_OBS},  θ = 2π/T_obs = {THETA:.5f}')

In [ ]:
_b0 = df['s01'].to_numpy() < THETA
_b1 = df['s02'].to_numpy() < THETA
_b2 = df['s12'].to_numpy() < THETA
_raw_code = _b0.astype(np.int8) | (_b1.astype(np.int8) << 1) | (_b2.astype(np.int8) << 2)
_popcount = _b0.astype(np.int8) + _b1.astype(np.int8) + _b2.astype(np.int8)
# ≥2 бита из 3 -> «полная» (см. комментарий выше про abs()); иначе один конкретный бит.
df['code'] = np.select(
    [_popcount >= 2, _raw_code == 1, _raw_code == 2, _raw_code == 4],
    [4, 1, 2, 3], default=0
).astype(np.int8)

P_INPHASE = 0.9               # порог P, выше которого запуск считаем синфазным
df['p_hi'] = df['P'] > P_INPHASE

@lru_cache(maxsize=None)
def sub_with_code(eps):
    return df[df['eps'] == eps]

# Один проход агрегации по точкам сетки (delta1, delta2, eps, тип связи) —
# дальше все карты слайсят PTS, а не сканируют df заново.
KEYS = ['delta1', 'delta2', 'eps', 'coupling_type']
PTS = df.groupby(KEYS).agg(
    L_max=('L', 'max'), L_min=('L', 'min'),
    A_max=('A', 'max'), A_min=('A', 'min'),
    P_max=('P', 'max'), P_min=('P', 'min'),
    inphase_frac=('p_hi', 'mean'),
    n_codes=('code', 'nunique'),
).reset_index()
for _v in ('L', 'A', 'P'):
    PTS[f'{_v}_spread'] = PTS[f'{_v}_max'] - PTS[f'{_v}_min']
PTS['multi'] = PTS['n_codes'] > 1

# Доминирующий код, чистота моды и разбивка по кодам для ховера.
_counts = df.groupby(KEYS + ['code']).size().rename('n').reset_index()
_dom = (_counts.sort_values('n').drop_duplicates(KEYS, keep='last')
        .rename(columns={'code': 'dominant_code'})[KEYS + ['dominant_code']])
PTS = PTS.merge(_dom, on=KEYS)

_pt = _counts.groupby(KEYS)['n'].agg(dom_n='max', tot_n='sum')
_pt['agree_frac'] = _pt['dom_n'] / _pt['tot_n']
PTS = PTS.merge(_pt[['agree_frac']].reset_index(), on=KEYS)

_counts['lbl'] = np.array(REGIME_LABELS)[_counts['code'].to_numpy()] + ': ' + _counts['n'].astype(str)
_brk = (_counts.sort_values('n', ascending=False).groupby(KEYS)['lbl']
        .agg('<br>'.join).rename('breakdown').reset_index())
PTS = PTS.merge(_brk, on=KEYS)

def discrete_colorscale(colors):
    n = len(colors)
    return [pt for i, c in enumerate(colors) for pt in ([i / n, c], [(i + 1) / n, c])]

REGIME_SCALE = discrete_colorscale(REGIME_COLORS)

def _eps_step(name):
    return dict(method='animate', label=name,
                args=[[name], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                                   transition=dict(duration=0))])


def add_eps_slider(fig, frames, prefix='ε = ', pad_t=60):
    fig.frames = list(frames)
    fig.update_layout(sliders=[dict(active=0, pad={'t': pad_t},
                                    currentvalue={'prefix': prefix},
                                    steps=[_eps_step(f.name) for f in fig.frames])])
    return fig


def lock_square(fig, ncols, nrows=1):
    # В subplot'ах у каждой панели своя пара осей (x/y, x2/y2, ...), поэтому общий
    # update_yaxes(scaleanchor='x') привязал бы все Y к первой X. Якорим попанельно:
    # δ₁ и δ₂ пробегают один диапазон, карта обязана быть квадратной.
    for r in range(1, nrows + 1):
        for c in range(1, ncols + 1):
            k = (r - 1) * ncols + c
            fig.update_yaxes(scaleanchor=('x' if k == 1 else f'x{k}'), scaleratio=1,
                             constrain='domain', row=r, col=c)
    return fig


def add_eps_coupling_sliders(fig, frames, group_size, coupling_names, extra_visible=0):
    fig.frames = list(frames)
    total = len(coupling_names) * group_size
    coupling_steps = [dict(method='restyle', label=name,
                           args=[{'visible': [(ci * group_size <= i < (ci + 1) * group_size)
                                              if i < total else True
                                              for i in range(total + extra_visible)]}])
                      for ci, name in enumerate(coupling_names)]
    fig.update_layout(sliders=[
        dict(active=0, yanchor='top', y=0, pad={'t': 40},
             currentvalue={'prefix': 'ε = '}, steps=[_eps_step(f.name) for f in fig.frames]),
        dict(active=0, yanchor='top', y=0, pad={'t': 110},
             currentvalue={'prefix': 'тип связи: '}, steps=coupling_steps),
    ])
    return fig

# --- Размер шрифта в экспортируемых фигурах -----------------------------------
# Kaleido пишет PDF в пунктах: 1 px канвы = 1 pt. В статье фигура вставляется как
# \includegraphics[width=\textwidth], то есть ужимается с width до TEXTWIDTH_PT,
# и вместе с ней ужимается шрифт. Чтобы в печати получить TARGET_PT, задаём на
# канве шрифт, увеличенный в (width / TEXTWIDTH_PT) раз. Для сеток из многих
# панелей 10 pt получается слишком плотно, поэтому держим подписи немного мельче.
TEXTWIDTH_PT = 484          # \textwidth статьи: A4 210мм минус поля 25+15 => 170мм
TARGET_PT = 8               # желаемый кегль подписей в готовом PDF


def fit_fonts(fig, width):
    k = width / TEXTWIDTH_PT
    base = TARGET_PT * k
    fig.update_layout(font=dict(size=base))
    # Заголовки панелей и подписи рядов — это annotations, им нужен свой размер.
    for ann in (fig.layout.annotations or []):
        ann.font = dict(size=base * 0.95)
    for ax in fig.layout:
        if ax.startswith(('xaxis', 'yaxis')):
            fig.layout[ax].title.font = dict(size=base)
            fig.layout[ax].tickfont = dict(size=base * 0.85)
    for tr in fig.data:
        cb = getattr(tr, 'colorbar', None)
        if cb is not None:
            cb.title.font = dict(size=base)
            cb.tickfont = dict(size=base * 0.85)
    return fig


# Длинные названия связей не влезают в узкую колонку — переносим по строкам.
COUPLING_NAMES_BR = [n.replace(' (', '<br>(') for n in COUPLING_NAMES]


## Подбор порога θ

### Распределение наклонов разности фаз

In [ ]:
# Распределение |s| по реальным рёбрам звезды (s01, s02). s12 не включаем:
# он не независим (s01 = s02 - s12 тождественно), иначе в гистограмму подмешивается
# производная величина.
# Обе оси логарифмические: запертая популяция живёт в 1e-6..1e-3, и на линейной
# шкале [0, 0.2] она целиком укладывается в первые полпроцента оси — провал не виден.
_BINS = np.logspace(-6, 0, 73)
_CTR = np.sqrt(_BINS[:-1] * _BINS[1:])
_DLOG = np.diff(np.log10(_BINS))


def slope_hist(eps, ct):
    g = df[(df['eps'] == eps) & (df['coupling_type'] == ct)]
    s = np.concatenate([g['s01'].to_numpy(), g['s02'].to_numpy()])
    s = s[s > 0]
    h, _ = np.histogram(s, bins=_BINS)
    return h / h.sum() / _DLOG          # плотность на порядок величины (на единицу log10|s|)


def hist_traces_slope(eps):
    return [go.Scatter(x=_CTR, y=slope_hist(eps, ct), mode='lines',
                       line=dict(color='steelblue', shape='hv'), fill='tozeroy')
            for ct in range(4)]


fig = make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES_BR,
                    horizontal_spacing=0.03, shared_yaxes=True)
for i, tr in enumerate(hist_traces_slope(epsilons[0])):
    fig.add_trace(tr, row=1, col=i + 1)

frames = [go.Frame(name=str(e), data=hist_traces_slope(e)) for e in epsilons]
fig.update_xaxes(type='log', title_text='|s|', range=[-6, 0])
fig.update_yaxes(type='log', range=[-2.5, 0.7], col=1, title_text='плотность на порядок величины')
fig.update_yaxes(type='log', range=[-2.5, 0.7])
for c in range(1, 5):
    fig.add_vline(x=THETA, line=dict(color='green', dash='dash', width=2), row=1, col=c)
fig.update_layout(height=400, width=1300, showlegend=False,
                  title=f'Распределение наклонов s01, s02 (зелёная линия — θ = 2π/T_obs = {THETA:.4f})')
add_eps_slider(fig, frames)
fig.show()

In [ ]:
print(f'θ = 2π/T_obs = {THETA:.5f}   (единая для всех ε и всех типов связи)')

### Устойчивость карты к выбору θ

θ задана физически: $\theta = 2\pi/T_{obs}$ — порог, выше которого «захват» допускал бы
полный проскок фазы на $2\pi$ за окно наблюдения. Автоподбор по плато убран.

Проверка: доля области захвата, меняющая режим при **четырёхкратном** сдвиге порога
($\theta/2 \to 2\theta$). Знаменатель — ячейки, запертые хоть при одном из порогов
(фон, который никогда не заперт, разбавляет метрику и исключён).

In [ ]:
def dominant_at(g, th, idx):
    code = ((g['s01'].to_numpy() < th).astype(np.int8)
            | ((g['s02'].to_numpy() < th).astype(np.int8) << 1))
    cnt = g.assign(code=code).groupby(['delta1', 'delta2', 'code']).size().rename('n').reset_index()
    s = (cnt.sort_values('n').drop_duplicates(['delta1', 'delta2'], keep='last')
            .set_index(['delta1', 'delta2'])['code'])
    return s.reindex(idx).to_numpy()


_rows = []
for eps in epsilons:
    for ct in range(4):
        g = df[(df['eps'] == eps) & (df['coupling_type'] == ct)]
        idx = pd.MultiIndex.from_frame(g[['delta1', 'delta2']].drop_duplicates())
        lo = dominant_at(g, THETA / 2, idx)
        hi = dominant_at(g, THETA * 2, idx)
        active = (lo > 0) | (hi > 0)              # заперта хоть при одном из порогов
        flip = (lo != hi) & active
        _rows.append({'eps': eps, 'coupling': COUPLING_NAMES[ct],
                      'flip_frac': flip.sum() / active.sum() if active.sum() else np.nan})

THETA_ROBUST = (pd.DataFrame(_rows)
                .pivot(index='eps', columns='coupling', values='flip_frac')[COUPLING_NAMES])
THETA_ROBUST.style.format('{:.1%}').background_gradient(cmap='Reds', vmin=0, vmax=0.7)

### Карта наклона и устойчивость к θ

`min|slope|` по н.у. (плато ≈ 0 — захват) + контуры θ ∈ {0.01…0.05}. Контуры сбиты в тонкую полосу ⇒ граница захвата не зависит от выбора θ.

In [ ]:
SLOPE_MIN = df.groupby(KEYS)[['s01', 's02', 's12']].min().reset_index()
_PAIRS = [('s01', '0–1'), ('s02', '0–2'), ('s12', '1–2')]


def slope_traces(eps):
    a = SLOPE_MIN[SLOPE_MIN['eps'] == eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        for pi, (col, _) in enumerate(_PAIRS):
            piv = d.pivot(index='delta1', columns='delta2', values=col)
            traces.append(go.Heatmap(
                z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                colorscale='Viridis', zmin=0, zmax=0.06, showscale=(pi == 0),
                colorbar=dict(title='мин. |s|', x=1.02, len=0.9, thickness=12),
                hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.4f}<extra></extra>'))
            traces.append(go.Contour(
                z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                contours=dict(start=0.01, end=0.05, size=0.01, coloring='lines'),
                line=dict(width=1, color='white'), showscale=False, hoverinfo='skip'))
    return traces


fig = make_subplots(rows=1, cols=3, subplot_titles=[lbl for _, lbl in _PAIRS], horizontal_spacing=0.06)
for idx, tr in enumerate(slope_traces(epsilons[0])):
    tr.visible = (idx // 6 == 0)
    panel = (idx % 6) // 2
    fig.add_trace(tr, row=1, col=panel + 1)

frames = [go.Frame(name=str(e), data=slope_traces(e)) for e in epsilons]
fig.update_xaxes(title_text='δ₂')
fig.update_yaxes(title_text='δ₁', col=1)
lock_square(fig, 3)
fig.update_layout(height=450, width=1300, margin=dict(b=140), title='Минимальный по н.у. наклон |s| и контуры θ')
add_eps_coupling_sliders(fig, frames, group_size=6, coupling_names=COUPLING_NAMES)
fig.show()

## Карты режимов синхронизации

In [ ]:
all_agg = {e: PTS[PTS['eps'] == e] for e in epsilons}

In [ ]:
_LBL_ARR = np.array(REGIME_LABELS)


def add_regime_legend(fig):
    # Дискретная heatmap не создаёт нормальную легенду категорий сама.
    for lbl, col in zip(REGIME_LABELS, REGIME_COLORS):
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers', name=lbl,
            marker=dict(size=12, symbol='square', color=col,
                        line=dict(width=0.5, color='#888')),
            showlegend=True, hoverinfo='skip'), row=1, col=1)
    return fig


def regime_traces(eps):
    a = all_agg[eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        dom = d.pivot(index='delta1', columns='delta2', values='dominant_code')
        brk = d.pivot(index='delta1', columns='delta2', values='breakdown')
        agree = d.pivot(index='delta1', columns='delta2', values='agree_frac')
        label = _LBL_ARR[dom.values.astype(int)]
        outside_mode = 1 - agree.values
        custom = np.dstack([label, brk.values, outside_mode])
        traces.append(go.Heatmap(
            z=dom.values, x=dom.columns.tolist(), y=dom.index.tolist(),
            customdata=custom, colorscale=REGIME_SCALE, zmin=-0.5, zmax=4.5, showscale=False,
            hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>мода: %{customdata[0]}'
                          '<br>вне моды: %{customdata[2]:.3f}'
                          '<br>%{customdata[1]}<extra></extra>'))
        traces.append(go.Heatmap(
            z=outside_mode, x=agree.columns.tolist(), y=agree.index.tolist(),
            colorscale='Viridis', zmin=0, zmax=DIS_MAX, showscale=True,
            colorbar=dict(title='вне моды', x=1.01, len=0.78, thickness=12),
            hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>вне моды: %{z:.3f}<extra></extra>'))
    return traces


def layout_regime(fig, title):
    fig.update_xaxes(title_text='δ₂')
    fig.update_yaxes(title_text='δ₁', col=1)
    lock_square(fig, 2)
    fig.update_layout(
        plot_bgcolor='white', margin=dict(t=95, b=150, r=260),
        showlegend=True,
        legend=dict(orientation='v', x=1.10, y=0.98, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0)', title_text='режим'),
        title=title)
    return fig


fig = make_subplots(rows=1, cols=2, subplot_titles=['мода режима', 'доля н.у. вне моды'],
                    horizontal_spacing=0.12)
for idx, tr in enumerate(regime_traces(epsilons[0])):
    tr.visible = (idx // 2 == 0)
    fig.add_trace(tr, row=1, col=idx % 2 + 1)
add_regime_legend(fig)

frames = [go.Frame(name=str(e), data=regime_traces(e)) for e in epsilons]

fig.update_layout(height=620, width=1350, autosize=True)
layout_regime(fig, 'Режимы синхронизации: мода режима и доля н.у. вне моды')
add_eps_coupling_sliders(fig, frames, group_size=2, coupling_names=COUPLING_NAMES,
                         extra_visible=len(REGIME_LABELS))
fig.show()

## Целевые функции

In [ ]:
A_EDGES = np.linspace(df['A'].min(), df['A'].max(), 61)
P_EDGES = np.linspace(df['P'].min(), df['P'].max(), 61)
_AC = 0.5 * (A_EDGES[:-1] + A_EDGES[1:])
_PC = 0.5 * (P_EDGES[:-1] + P_EDGES[1:])

_PA_HIST = {}
for (e, ct), g in df.groupby(['eps', 'coupling_type']):
    h, _, _ = np.histogram2d(g['A'].values, g['P'].values, bins=[A_EDGES, P_EDGES])
    _PA_HIST[(e, ct)] = np.log1p(h.T)


def pa_density(eps):
    return [go.Heatmap(z=_PA_HIST[(eps, ct)], x=_AC, y=_PC,
                       colorscale='Blues', showscale=False,
                       hovertemplate='A=%{x:.3f} P=%{y:.3f}<br>log(1+n)=%{z:.2f}<extra></extra>')
            for ct in range(4)]


fig = make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES_BR,
                    horizontal_spacing=0.04, shared_yaxes=True)
for i, tr in enumerate(pa_density(epsilons[0])):
    fig.add_trace(tr, row=1, col=i + 1)

frames = [go.Frame(name=str(e), data=pa_density(e)) for e in epsilons]
fig.update_xaxes(title_text='A')
fig.update_yaxes(title_text='P', col=1)
fig.update_layout(height=400, width=1300, title='Совместное распределение P и A (логарифм плотности запусков)')
add_eps_slider(fig, frames)
fig.show()

In [ ]:
def _corrs(g):
    return pd.Series({'ρ_Pearson': g['A'].corr(g['P']),
                      'ρ_Spearman': g['A'].corr(g['P'], method='spearman')})


def pa_corr_table(mask, label):
    corr = (df.loc[mask]
              .groupby(['eps', 'coupling_type'])[['A', 'P']]
              .apply(_corrs)
              .reset_index())
    corr['coupling'] = corr['coupling_type'].map(dict(enumerate(COUPLING_NAMES)))
    corr['subset'] = label
    return corr


PA_CORR_CAPTURED = pa_corr_table(df['code'] != 0, 'в захвате')
PA_CORR_ALL = pa_corr_table(np.ones(len(df), dtype=bool), 'вся карта')
PA_CORR = pd.concat([PA_CORR_CAPTURED, PA_CORR_ALL], ignore_index=True)

_m_cap = PA_CORR_CAPTURED.pivot(index='eps', columns='coupling', values='ρ_Spearman')[COUPLING_NAMES]
_m_all = PA_CORR_ALL.pivot(index='eps', columns='coupling', values='ρ_Spearman')[COUPLING_NAMES]
fig = make_subplots(rows=1, cols=2, subplot_titles=['в захвате', 'вся карта'],
                    horizontal_spacing=0.10, shared_yaxes=True)
for col, m in enumerate([_m_cap, _m_all], start=1):
    fig.add_trace(go.Heatmap(
        z=m.values, x=m.columns.tolist(), y=[f'{e:.2f}' for e in m.index],
        text=m.values, texttemplate='%{text:.3f}',
        colorscale='RdYlGn', zmin=float(min(_m_cap.values.min(), _m_all.values.min())), zmax=1.0,
        colorbar=dict(title='ρ Спирмена', x=1.02) if col == 2 else None,
        showscale=(col == 2)), row=1, col=col)
fig.update_layout(height=430, width=1250, title='Корреляция Спирмана A и P по ε и типу связи',
                  margin=dict(t=80, b=90, l=70, r=120))
fig.update_xaxes(title_text='тип связи')
fig.update_yaxes(title_text='ε', col=1)
fig.show()

## Синфазность

Две величины по начальным условиям в каждой точке сетки:

- **max $P$** — достигается ли синфазный режим хоть при каких-то н.у.;
- **$\Delta P$ = max − min** — насколько синфазность зависит от выбора н.у.

Медиана и квантили здесь неинформативны: если синфазный режим существует, но узок по
бассейну притяжения, они его просто не видят.

In [ ]:
P_LO = float(PTS['P_min'].min())
P_SPREAD_HI = float(PTS['P_spread'].max())


def inphase_heatmap(piv, zmin, zmax, cbar_title, cbar_x):
    return go.Heatmap(z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
                      colorscale='Viridis', zmin=zmin, zmax=zmax, showscale=True,
                      colorbar=dict(title=cbar_title, x=cbar_x, len=0.9, thickness=12),
                      hovertemplate='δ₁=%{y:.3f} δ₂=%{x:.3f}<br>%{z:.3f}<extra></extra>')


def inphase_traces(eps):
    a = PTS[PTS['eps'] == eps]
    traces = []
    for ct in range(4):
        d = a[a['coupling_type'] == ct]
        pmax = d.pivot(index='delta1', columns='delta2', values='P_max')
        pspr = d.pivot(index='delta1', columns='delta2', values='P_spread')
        # Pmax — есть ли синфазный режим хоть при каких-то н.у. в этой точке.
        traces.append(inphase_heatmap(pmax, P_LO, 1.0, 'макс. P', 0.44))
        # Pmax - Pmin — насколько синфазность зависит от начальных условий в этой точке.
        traces.append(inphase_heatmap(pspr, 0.0, P_SPREAD_HI, 'разброс P', 1.0))
    return traces


fig = make_subplots(rows=1, cols=2, subplot_titles=['максимум P по н.у.', 'разброс P: максимум − минимум'],
                    horizontal_spacing=0.18)
for idx, tr in enumerate(inphase_traces(epsilons[0])):
    tr.visible = (idx // 2 == 0)
    fig.add_trace(tr, row=1, col=idx % 2 + 1)

frames = [go.Frame(name=str(e), data=inphase_traces(e)) for e in epsilons]
fig.update_xaxes(title_text='δ₂')
fig.update_yaxes(title_text='δ₁', col=1)
lock_square(fig, 2)
fig.update_layout(height=600, width=1200, margin=dict(b=140),
                  title='Синфазность в точке: максимум и разброс P по 30 начальным условиям')
add_eps_coupling_sliders(fig, frames, group_size=2, coupling_names=COUPLING_NAMES)
fig.show()


## Исследование $\Delta L$, $\Delta A$, $\Delta P$ как критериев мультистабильности

In [ ]:
_SPREADS = [('L_spread', 'ΔL'), ('A_spread', 'ΔA'), ('P_spread', 'ΔP')]
_REG = [(False, 'один режим', 'steelblue'), (True, 'мультистаб.', 'tomato')]


def hist_traces(agg):
    traces = []
    for si, (col, _) in enumerate(_SPREADS):
        for ct in range(4):
            d = agg[agg['coupling_type'] == ct]
            for is_multi, name, color in _REG:
                traces.append(go.Histogram(
                    x=d.loc[d['multi'] == is_multi, col], nbinsx=50,
                    marker_color=color, opacity=0.6, name=name, legendgroup=name,
                    showlegend=(si == 0 and ct == 0)))
    return traces


fig = make_subplots(rows=3, cols=4, row_titles=[lbl for _, lbl in _SPREADS],
                    column_titles=COUPLING_NAMES_BR, horizontal_spacing=0.05, vertical_spacing=0.07)
it = iter(hist_traces(PTS[PTS['eps'] == epsilons[0]]))
for si in range(3):
    for ct in range(4):
        for _ in _REG:
            fig.add_trace(next(it), row=si + 1, col=ct + 1)
fig.update_yaxes(type='log', nticks=4)
fig.update_xaxes(title_text='', nticks=4)
fig.update_yaxes(title_text='')
fig.add_annotation(text='разброс по 30 н.у.', x=0.5, y=-0.08, xref='paper', yref='paper',
                   showarrow=False, xanchor='center', yanchor='top')
fig.add_annotation(text='число точек сетки', x=-0.06, y=0.5, xref='paper', yref='paper',
                   showarrow=False, textangle=-90, xanchor='center', yanchor='middle')

frames = [go.Frame(name=str(e), data=hist_traces(PTS[PTS['eps'] == e])) for e in epsilons]
fig.update_layout(height=660, width=1300, barmode='overlay', title='Разброс L, A, P в одно- и мультистабильных точках',
                  margin=dict(l=130, r=150, t=80, b=125))
add_eps_slider(fig, frames)
fig.show()

## Автоматический экспорт фигур без слайдеров

Следующая ячейка экспортирует статические фигуры в PDF (для Overleaf: `\includegraphics`
понимает их без дополнительных пакетов) и в SVG (для просмотра). В каждом файле
зафиксированы конкретные $ε$ и/или тип связи, поэтому интерактивные слайдеры не нужны.

Экспорт требует пакет `kaleido`. Если ячейка падает с ошибкой про Kaleido, установи его
в окружение `display` (`uv add kaleido`) и перезапусти экспорт.

In [ ]:
EXPORT_DIR = Path('../plots/export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def _eps_tag(eps):
    return f"eps-{float(eps):.2f}".replace('.', 'p')


def _ct_tag(ct):
    return COUPLING_SLUGS[ct]


def _write_static(fig, name):
    fit_fonts(fig, fig.layout.width or TEXTWIDTH_PT)
    # PDF — для Overleaf: pdflatex вставляет его \includegraphics'ом без доп. пакетов.
    # SVG — для просмотра в браузере. В обоих heatmap остаётся растром (kaleido
    # встраивает её как <image>), векторными выходят оси, подписи и легенда.
    pdf = EXPORT_DIR / f'{name}.pdf'
    fig.write_image(pdf, format='pdf')

    svg = EXPORT_DIR / f'{name}.svg'
    fig.write_image(svg, format='svg')
    # kaleido начинает SVG сразу с <svg>, без XML-декларации. Байты — UTF-8, но без
    # объявления кодировки просмотрщик угадывает cp1252, и δ/− превращаются в кракозябры.
    svg.write_bytes(b'<?xml version="1.0" encoding="utf-8"?>\n' + svg.read_bytes())
    return pdf


def fig_slope_hist_static(eps):
    fig = make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES_BR,
                        horizontal_spacing=0.03, shared_yaxes=True)
    for i, tr in enumerate(hist_traces_slope(eps)):
        fig.add_trace(tr, row=1, col=i + 1)
    fig.update_xaxes(type='log', title_text='|s|', range=[-6, 0])
    fig.update_yaxes(type='log', range=[-2.5, 0.7], col=1, title_text='плотность на порядок величины')
    fig.update_yaxes(type='log', range=[-2.5, 0.7])
    for c in range(1, 5):
        fig.add_vline(x=THETA, line=dict(color='green', dash='dash', width=2), row=1, col=c)
    fig.update_layout(height=400, width=1300, showlegend=False, margin=dict(t=75))
    return fig


def fig_slope_map_static(eps, ct):
    fig = make_subplots(rows=1, cols=3, subplot_titles=[lbl for _, lbl in _PAIRS], horizontal_spacing=0.06)
    for idx, tr in enumerate(slope_traces(eps)[ct * 6:(ct + 1) * 6]):
        panel = (idx % 6) // 2
        fig.add_trace(tr, row=1, col=panel + 1)
    fig.update_xaxes(title_text='δ₂')
    fig.update_yaxes(title_text='δ₁', col=1)
    lock_square(fig, 3)
    fig.update_layout(height=450, width=1300, margin=dict(t=40))
    return fig


def fig_regime_static(eps, ct):
    # Заголовок не рисуем: в статье его роль играет \caption, дублировать нельзя.
    fig = make_subplots(rows=1, cols=2, subplot_titles=['мода режима', 'доля н.у. вне моды'],
                        horizontal_spacing=0.12)
    for idx, tr in enumerate(regime_traces(eps)[ct * 2:(ct + 1) * 2]):
        fig.add_trace(tr, row=1, col=idx + 1)
    add_regime_legend(fig)
    fig.update_layout(height=620, width=1350, autosize=False, margin=dict(t=40))
    layout_regime(fig, None)
    return fig


def fig_pa_density_static(eps):
    fig = make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES_BR,
                        horizontal_spacing=0.04, shared_yaxes=True)
    for i, tr in enumerate(pa_density(eps)):
        fig.add_trace(tr, row=1, col=i + 1)
    fig.update_xaxes(title_text='A')
    fig.update_yaxes(title_text='P', col=1)
    fig.update_layout(height=400, width=1300, margin=dict(t=75))
    return fig


def fig_inphase_static(eps, ct):
    fig = make_subplots(rows=1, cols=2, subplot_titles=['максимум P по н.у.', 'разброс P: максимум − минимум'],
                        horizontal_spacing=0.18)
    for idx, tr in enumerate(inphase_traces(eps)[ct * 2:(ct + 1) * 2]):
        fig.add_trace(tr, row=1, col=idx + 1)
    fig.update_xaxes(title_text='δ₂')
    fig.update_yaxes(title_text='δ₁', col=1)
    lock_square(fig, 2)
    # Заголовок не рисуем: в статье его роль играет \caption.
    fig.update_layout(height=600, width=1200, margin=dict(t=40))
    return fig


def fig_spread_hist_static(eps):
    fig = make_subplots(rows=3, cols=4, row_titles=[lbl for _, lbl in _SPREADS],
                        column_titles=COUPLING_NAMES_BR, horizontal_spacing=0.05, vertical_spacing=0.07)
    it = iter(hist_traces(PTS[PTS['eps'] == eps]))
    for si in range(3):
        for ct in range(4):
            for _ in _REG:
                fig.add_trace(next(it), row=si + 1, col=ct + 1)
    fig.update_yaxes(type='log', nticks=4)
    fig.update_xaxes(title_text='', nticks=4)
    fig.update_yaxes(title_text='')
    fig.add_annotation(text='разброс по 30 н.у.', x=0.5, y=-0.08, xref='paper', yref='paper',
                       showarrow=False, xanchor='center', yanchor='top')
    fig.add_annotation(text='число точек сетки', x=-0.06, y=0.5, xref='paper', yref='paper',
                       showarrow=False, textangle=-90, xanchor='center', yanchor='middle')
    fig.update_layout(height=660, width=1300, barmode='overlay', margin=dict(l=130, r=150, t=75, b=125))
    return fig


# Фигуры без слайдеров — по одному файлу на всю фигуру.
def fig_pa_corr_static():
    m_cap = PA_CORR_CAPTURED.pivot(index='eps', columns='coupling', values='ρ_Spearman')[COUPLING_NAMES]
    m_all = PA_CORR_ALL.pivot(index='eps', columns='coupling', values='ρ_Spearman')[COUPLING_NAMES]
    zmin = float(min(m_cap.values.min(), m_all.values.min()))
    fig = make_subplots(rows=1, cols=2, subplot_titles=['в захвате', 'вся карта'],
                        horizontal_spacing=0.10, shared_yaxes=True)
    for col, m in enumerate([m_cap, m_all], start=1):
        fig.add_trace(go.Heatmap(
            z=m.values, x=m.columns.tolist(), y=[f'{e:.2f}' for e in m.index],
            text=m.values, texttemplate='%{text:.3f}',
            colorscale='RdYlGn', zmin=zmin, zmax=1.0,
            colorbar=dict(title='ρ Спирмена', x=1.02) if col == 2 else None,
            showscale=(col == 2)), row=1, col=col)
    fig.update_xaxes(title_text='тип связи')
    fig.update_yaxes(title_text='ε', col=1)
    fig.update_layout(height=430, width=1250, margin=dict(t=70, b=90, l=70, r=120))
    return fig


def fig_timeseries_static():
    # Ячейка с временными рядами идёт ниже по ноутбуку, поэтому параметры задаём
    # здесь локально — иначе экспорт при Run All падает с NameError.
    t_trans, window = 600.0, 40.0
    runs = [
        {'title': 'δ₁=-0.20 δ₂=0.12 — код 2 (0↔2), L≈0.834', 'path': '../simulation/demo/small_l_code2.csv'},
        {'title': 'δ₁=-0.20 δ₂=0.12 — код 0 (нет синхр.), L≈0.838', 'path': '../simulation/demo/small_l_code0.csv'},
        {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈2.1', 'path': '../simulation/demo/large_dl_high_l.csv'},
        {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈0.235', 'path': '../simulation/demo/large_dl_low_l.csv'},
    ]
    colors = {'x0': '#e41a1c', 'x1': '#4daf4a', 'x2': '#377eb8'}
    fig = make_subplots(rows=len(runs), cols=1, shared_xaxes=True,
                        subplot_titles=[r['title'] for r in runs])
    for ri, run in enumerate(runs):
        dfr = pd.read_csv(run['path'], comment='#')
        steady = dfr[(dfr['t'] >= t_trans) & (dfr['t'] < t_trans + window)]
        for col, c in colors.items():
            fig.add_trace(go.Scatter(x=steady['t'], y=steady[col], mode='lines',
                                     line=dict(color=c, width=1), name=col,
                                     legendgroup=col, showlegend=(ri == 0)),
                          row=ri + 1, col=1)
    fig.update_xaxes(title_text='t', row=len(runs), col=1)
    fig.update_yaxes(title_text='x')
    fig.update_layout(height=1000, width=1300, margin=dict(t=40))
    return fig


# Окно зума в центр. Верхняя граница шире нижней: у ненормированной связи центр
# языка захвата сидит не в нуле, а в δ ≈ ε/2 (сдвиг от степени узла).
ZOOM = (-0.10, 0.15)


# Сводные фигуры: все пять ε в одной картинке. Колонки — ε, строки — панели.
# Так в статье не приходится произвольно выбирать пару значений ε для иллюстрации.
def fig_regime_alleps_static(ct):
    # Оси общие: подписи и деления только по краям сетки, иначе при крупном шрифте
    # (fit_fonts масштабирует его под ширину канвы) они наползают друг на друга.
    fig = make_subplots(
        rows=2, cols=len(epsilons),
        subplot_titles=[f'ε = {e:.2f}' for e in epsilons] + [''] * len(epsilons),
        shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=0.02, vertical_spacing=0.07)
    for j, eps in enumerate(epsilons):
        dom, out = regime_traces(eps)[ct * 2:(ct + 1) * 2]
        out.showscale = (j == len(epsilons) - 1)
        out.colorbar = dict(title='вне моды', x=1.02, y=0.25, len=0.45, thickness=12)
        fig.add_trace(dom, row=1, col=j + 1)
        fig.add_trace(out, row=2, col=j + 1)
    add_regime_legend(fig)
    lock_square(fig, len(epsilons), nrows=2)
    fig.update_xaxes(title_text='δ₂', row=2)
    fig.update_yaxes(title_text='δ₁', col=1)
    fig.update_layout(height=700, width=1700, autosize=False, plot_bgcolor='white',
                      margin=dict(t=60, b=150, r=170), showlegend=True,
                      legend=dict(orientation='h', x=0.5, y=-0.22, xanchor='center',
                                  bgcolor='rgba(255,255,255,0)', title_text='режим'))
    return fig


def fig_inphase_alleps_static(ct):
    fig = make_subplots(
        rows=2, cols=len(epsilons),
        subplot_titles=[f'ε = {e:.2f}' for e in epsilons] + [''] * len(epsilons),
        shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=0.02, vertical_spacing=0.07)
    for j, eps in enumerate(epsilons):
        pmax, pspr = inphase_traces(eps)[ct * 2:(ct + 1) * 2]
        pmax.showscale = (j == len(epsilons) - 1)
        pmax.colorbar = dict(title='макс. P', x=1.02, y=0.78, len=0.42, thickness=12)
        pspr.showscale = (j == len(epsilons) - 1)
        pspr.colorbar = dict(title='разброс P', x=1.02, y=0.24, len=0.42, thickness=12)
        fig.add_trace(pmax, row=1, col=j + 1)
        fig.add_trace(pspr, row=2, col=j + 1)
    lock_square(fig, len(epsilons), nrows=2)
    fig.update_xaxes(title_text='δ₂', row=2)
    fig.update_yaxes(title_text='δ₁', col=1)
    fig.update_layout(height=700, width=1700, autosize=False, margin=dict(t=60, b=70, r=190))
    return fig


def _zoomed(fig):
    fig = go.Figure(fig)
    fig.update_xaxes(range=list(ZOOM))
    fig.update_yaxes(range=list(ZOOM))
    # У фигур без заголовка (режимы, синфазность) не воскрешаем его: то, что это
    # центральная область, сказано в \caption.
    ttl = fig.layout.title.text
    if ttl:
        fig.update_layout(title=ttl + ' — центральная область')
    return fig


def export_all_static():
    exported = []
    for eps in epsilons:
        et = _eps_tag(eps)
        exported.append(_write_static(fig_slope_hist_static(eps), f'slope-hist-{et}'))
        exported.append(_write_static(fig_pa_density_static(eps), f'pa-density-{et}'))
        exported.append(_write_static(fig_spread_hist_static(eps), f'spread-hist-{et}'))
        for ct in range(4):
            tag = f'{et}-{_ct_tag(ct)}'
            exported.append(_write_static(fig_slope_map_static(eps, ct), f'slope-map-{tag}'))

            reg = fig_regime_static(eps, ct)
            inp = fig_inphase_static(eps, ct)
            exported.append(_write_static(reg, f'regime-{tag}'))
            exported.append(_write_static(inp, f'inphase-{tag}'))
            exported.append(_write_static(_zoomed(reg), f'regime-{tag}-zoom'))
            exported.append(_write_static(_zoomed(inp), f'inphase-{tag}-zoom'))

    # Сводные: все ε в одной фигуре, по одной на тип связи.
    for ct in range(4):
        st = _ct_tag(ct)
        reg_all = fig_regime_alleps_static(ct)
        inp_all = fig_inphase_alleps_static(ct)
        exported.append(_write_static(reg_all, f'regime-alleps-{st}'))
        exported.append(_write_static(inp_all, f'inphase-alleps-{st}'))
        exported.append(_write_static(_zoomed(reg_all), f'regime-alleps-{st}-zoom'))
    exported.append(_write_static(fig_pa_corr_static(), 'pa-corr'))
    exported.append(_write_static(fig_timeseries_static(), 'timeseries'))
    print(f'Exported {len(exported)} figures (PDF + SVG) to {EXPORT_DIR}')
    return exported


# При Run All экспорт выполняется автоматически. Если нужно только открыть ноутбук
# без долгой записи PNG, поставь RUN_STATIC_EXPORT = False.
RUN_STATIC_EXPORT = True
if RUN_STATIC_EXPORT:
    EXPORTED_STATIC = export_all_static()

In [ ]:
def runs_at(eps, ct, d1, d2):
    s = sub_with_code(eps)
    return s[(s['coupling_type'] == ct)
             & np.isclose(s['delta1'], d1) & np.isclose(s['delta2'], d2)]


def counterexamples(eps, ct=0):
    d = PTS[(PTS['eps'] == eps) & (PTS['coupling_type'] == ct)]
    return (d[d['multi'] & (d['L_spread'] < 0.1)].head(3),
            d[~d['multi'] & (d['L_spread'] > 0.5)].head(3))


def show_counterexamples(eps, ct=0):
    ms, ml = counterexamples(eps, ct)
    print(f'=== ε={eps}, {COUPLING_NAMES[ct]} ===')
    print('Мультистаб. с малым ΔL (<0.1):')
    for _, r in ms.iterrows():
        codes = sorted(runs_at(eps, ct, r.delta1, r.delta2)['code'].unique())
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.L_spread:.4f} коды={codes}')
    print('Один режим с большим ΔL (>0.5):')
    for _, r in ml.iterrows():
        L = runs_at(eps, ct, r.delta1, r.delta2)['L'].round(3).tolist()
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.L_spread:.4f} L={L}')


show_counterexamples(epsilons[2])

### Проверка точки с маленьким $\Delta L$, но мультистабильностью

In [ ]:
def show_runs_at(eps, kind, ct=0):
    ms, ml = counterexamples(eps, ct)
    sel = ms if kind == 'multi_small' else ml
    if len(sel) == 0:
        print('нет таких точек в данных'); return
    r0 = sel.iloc[0]
    pt = runs_at(eps, ct, r0.delta1, r0.delta2)
    print(f'ε={eps} δ1={r0.delta1:.2f} δ2={r0.delta2:.2f} ({COUPLING_NAMES[ct]}), ΔL={r0.L_spread:.4f}')
    print(pt[['code', 'L', 'x0', 'y0', 'x1', 'y1', 'x2', 'y2']].round(3).to_string(index=False))


show_runs_at(epsilons[2], 'multi_small')

### Проверка точки с высоким $\Delta L$, но однорежимностью

In [ ]:
show_runs_at(epsilons[2], 'mono_large')

In [ ]:
t_trans, window = 600.0, 40.0
runs = [
    {'title': 'δ₁=-0.20 δ₂=0.12 — код 2 (0↔2), L≈0.834', 'path': '../simulation/demo/small_l_code2.csv'},
    {'title': 'δ₁=-0.20 δ₂=0.12 — код 0 (нет синхр.), L≈0.838', 'path': '../simulation/demo/small_l_code0.csv'},
    {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈2.1', 'path': '../simulation/demo/large_dl_high_l.csv'},
    {'title': 'δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈0.235', 'path': '../simulation/demo/large_dl_low_l.csv'},
]
colors = {'x0': '#e41a1c', 'x1': '#4daf4a', 'x2': '#377eb8'}

fig = make_subplots(rows=len(runs), cols=1, shared_xaxes=True,
                    subplot_titles=[r['title'] for r in runs])
for ri, run in enumerate(runs):
    dfr = pd.read_csv(run['path'], comment='#')
    steady = dfr[(dfr['t'] >= t_trans) & (dfr['t'] < t_trans + window)]
    for col, c in colors.items():
        fig.add_trace(go.Scatter(x=steady['t'], y=steady[col], mode='lines',
                                 line=dict(color=c, width=1), name=col,
                                 legendgroup=col, showlegend=(ri == 0)),
                      row=ri + 1, col=1)

fig.update_xaxes(title_text='t', row=len(runs), col=1)
fig.update_layout(height=1000, title='Осцилляции в установившемся режиме')
fig.show()